# Orchestrator Test (Unified DataSource)

This notebook demonstrates the **current** codepath:
- Use `run_metadata_extraction` with a single `source` argument (file, directory, dict, or DataSource)
- No legacy `dataset_path`/`file_type` parameters
- Auto-detects multi-table datasets and adds `relationship_analyst` automatically


In [ ]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

print("Imports OK")


In [ ]:
from src.context.context_factory import create_context
single_source = os.path.join(BASE, 'data/biota_dataset/biota.csv')
data_context = create_context(
    source=single_source,
    name='biota_single'
)
metadata_standard=METADATA_STANDARDS['basic']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

In [ ]:

for i, step in enumerate(plan.steps):
    print("=" * 50)
    print(f"Step {i}: {step.task}")
    print(f"  Rationale: {step.rationale}")
    print(f"  Required Artifacts: {step.inputs}")
    print(f"  Produced Artifacts: {step.outputs}")

In [ ]:
from src.context.context_factory import create_context
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['basic']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

In [ ]:
for i, step in enumerate(plan.steps):
    print("=" * 50)
    print(f"Step {i}: {step.task}")
    print(f"  Rationale: {step.rationale}")
    print(f"  Required Artifacts: {step.inputs}")
    print(f"  Produced Artifacts: {step.outputs}")

## Single CSV example

In [ ]:
from src.orchestrator import run_metadata_extraction
single_source = os.path.join(BASE, 'data/biota_dataset/biota.csv')

result_single = run_metadata_extraction(
    source=single_source,
    metadata_standard=METADATA_STANDARDS['basic'],
    topology_name='default'
)

print(f"Success: {result_single.success}")
print(f"Steps completed: {result_single.steps_completed}/{result_single.plan_steps_count}")
print("Tables analyzed:", list(result_single.final_metadata.get('artifacts', {}).keys()))

## Multi-table CSV dataset

In [ ]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

result_multi = run_metadata_extraction(
    source=multi_source,
    metadata_standard=METADATA_STANDARDS['relational'],
    topology_name='default'  # relationship_analyst is auto-added
)

print(f"Success: {result_multi.success}")
print(f"Steps completed: {result_multi.steps_completed}/{result_multi.plan_steps_count}")
print("Tables analyzed:", list(result_multi.table_metadata.keys()))
print("Relationships found:", len(result_multi.relationships))

## Directory of CSVs

In [ ]:
dir_source = os.path.join(BASE, 'data/biota_dataset/')

dir_result = run_metadata_extraction(
    source=dir_source,
    metadata_standard=METADATA_STANDARDS['relational'],
    topology_name='default'
)

print(f"Success: {dir_result.success}")
print(f"Tables analyzed: {list(dir_result.table_metadata.keys())}")
print(f"Relationships: {len(dir_result.relationships)}")

## SQLite example (if you have a .sqlite file)

Uncomment and set the correct path to test.

In [ ]:
# sqlite_source = os.path.join(BASE, 'data/mydb.sqlite')
# sqlite_result = run_metadata_extraction(
#     source=sqlite_source,
#     metadata_standard=METADATA_STANDARDS['relational'],
#     topology_name='default'
# )
# print(f"Success: {sqlite_result.success}")
# print(f"Tables analyzed: {list(sqlite_result.table_metadata.keys())}")

In [ ]:
# =============================================================================
# Metadata Agent - Orchestrator Test Notebook
# =============================================================================
# This notebook demonstrates the multi-agent metadata extraction system.
#
# Features:
# - Plan Generation: LLM generates step-by-step extraction plan
# - Parallel Execution: Multiple players work on each step
# - Debate: Players critique and revise each other's work
# - Synthesis: Results are consolidated into final output
# =============================================================================

import logging
import os
import sys
from pprint import pprint

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Check config key
from src.config import get_config_summary
print(get_config_summary())

# Import orchestrator components
from src.orchestrator import Orchestrator
from src.orchestrator.schemas import Plan, PlanStep
from src.standards import METADATA_STANDARDS
from src.topology import EXECUTION_TOPOLOGIES
from src.players import PLAYER_CONFIGS

print("✅ All imports successful")

## 1. Configuration

Set up the parameters for plan generation and execution.

In [ ]:
# --- Configuration ---
DATASET_PATH = "../data/biota.csv"
FILE_TYPE = "CSV"
METADATA_STANDARD_NAME = "basic"  # Options: 'basic', 'dublin_core'
TOPOLOGY_NAME = "fast"  # Options: 'default', 'fast', 'thorough', 'single'

print("=" * 50)
print("Configuration")
print("=" * 50)
print(f"Dataset Path: {DATASET_PATH}")
print(f"File Type: {FILE_TYPE}")
print(f"Metadata Standard: {METADATA_STANDARD_NAME}")
print(f"Topology: {TOPOLOGY_NAME}")

# Show topology details
topology = EXECUTION_TOPOLOGIES[TOPOLOGY_NAME]
print(f"\nTopology Details:")
print(f"  - Players per step: {topology['players_per_step']}")
print(f"  - Debate rounds: {topology['debate_rounds']}")
print(f"  - Player pool: {topology['player_pool']}")

## 2. Explore Available Configurations

View available topologies and player configurations.

In [ ]:
# Show all available topologies
print("Available Execution Topologies:")
print("=" * 50)
for name, config in EXECUTION_TOPOLOGIES.items():
    print(f"\n{name}:")
    print(f"  Description: {config['description']}")
    print(f"  Players/step: {config['players_per_step']}")
    print(f"  Debate rounds: {config['debate_rounds']}")

In [ ]:
# Show all available player roles
print("Available Player Roles:")
print("=" * 50)
for name, config in PLAYER_CONFIGS.items():
    print(f"\n{name}:")
    print(f"  Role: {config['role_prompt'][:80]}...")
    tools = [t.name for t in config.get('tools', [])]
    print(f"  Tools: {tools if tools else 'None'}")

## 3. Plan Generation

Generate a step-by-step plan for metadata extraction using the Orchestrator.

In [ ]:
# Initialize the orchestrator
orchestrator = Orchestrator(topology_name=TOPOLOGY_NAME)

# Get the metadata standard content
metadata_standard = METADATA_STANDARDS[METADATA_STANDARD_NAME]
print("Metadata Standard:")
print(metadata_standard)

In [ ]:
# Generate the plan
print("Generating plan...")
print("=" * 50)

plan = orchestrator.generate_plan(
    file_type=FILE_TYPE,
    metadata_standard=metadata_standard
)

if plan:
    print("\n✅ Plan generated successfully!")
else:
    print("\n❌ Plan generation failed")

In [ ]:
# Inspect the generated plan
if plan:
    print("Generated Plan:")
    print("=" * 50)
    
    for i, step in enumerate(plan.steps):
        print(f"\nStep {i + 1}: {step.task}")
        print(f"  Player: {step.player}")
        print(f"  Rationale: {step.rationale}")
        print(f"  Inputs: {step.inputs}")
        print(f"  Outputs: {step.outputs}")
else:
    print("No plan to inspect. Run the plan generation cell first.")

In [ ]:
# View plan as dictionary (full details)
if plan:
    print("Plan as Dictionary:")
    print("=" * 50)
    pprint(plan.model_dump())

## 4. Plan Validation

Validate the plan's dataflow dependencies to ensure all inputs are available.

In [ ]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

## 5. Prepare Dataset

Ensure a test dataset exists for execution.

In [ ]:
# Check if dataset exists, create sample if not
import pandas as pd

if not os.path.exists(DATASET_PATH):
    print(f"Dataset not found at {DATASET_PATH}")
    print("Creating sample dataset...")
    
    os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)
    
    sample_df = pd.DataFrame({
        "id": [1, 2, 3, 4, 5],
        "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
        "age": [25, 30, 35, 28, 32],
        "city": ["NYC", "LA", "Chicago", "NYC", "Boston"],
        "salary": [50000, 60000, 75000, 55000, 80000]
    })
    sample_df.to_csv(DATASET_PATH, index=False)
    print(f"✅ Sample dataset created at {DATASET_PATH}")
else:
    print(f"✅ Dataset found at {DATASET_PATH}")

# Preview the dataset
df = pd.read_csv(DATASET_PATH)
print(f"\nDataset Preview ({len(df)} rows, {len(df.columns)} columns):")
print(df.head())

## 6. Full Execution

Execute the complete pipeline: plan generation + parallel players + debate.

**Note**: This will make multiple LLM calls and may take a few minutes.

In [ ]:
# Execute the plan with parallel players and debate
# This runs the full pipeline

if plan:
    print("Executing plan with parallel players and debate...")
    print("=" * 50)
    
    result = orchestrator.execute_plan(
        plan=plan,
        dataset_path=DATASET_PATH,
        metadata_standard=metadata_standard
    )
    
    if result.success:
        print("\n✅ Execution completed successfully!")
    else:
        print(f"\n❌ Execution failed: {result.error}")
else:
    print("No plan to execute. Run plan generation first.")

In [ ]:
# Inspect execution results
if 'result' in dir() and result:
    print("Execution Results Summary:")
    print("=" * 50)
    print(f"Success: {result.success}")
    print(f"Steps Completed: {result.steps_completed}/{result.plan_steps_count}")
    
    print("\n--- Step Results ---")
    for step_result in result.step_results:
        print(f"\nStep {step_result.step_index + 1}: {step_result.task}")
        print(f"  Player Role: {step_result.player_role}")
        print(f"  Success: {step_result.success}")
        print(f"  Debate Rounds: {step_result.debate_rounds_completed}")
        print(f"  Artifacts Produced: {list(step_result.artifacts.keys())}")
        if step_result.error:
            print(f"  Error: {step_result.error}")
else:
    print("No results to inspect. Run the execution cell first.")

In [ ]:
# View final workspace and metadata
if 'result' in dir() and result:
    print("Final Workspace Artifacts:")
    print("=" * 50)
    for name, value in result.final_workspace.items():
        print(f"\n--- {name} ---")
        # Truncate long values for display
        value_str = str(value)
        if len(value_str) > 500:
            print(value_str[:500] + "...")
        else:
            print(value_str)
    
    print("\n" + "=" * 50)
    print("Final Metadata:")
    print("=" * 50)
    pprint(result.final_metadata)
else:
    print("No results to display.")

In [ ]:
for key, value in result.final_metadata['artifacts'].items():
    print(f"\n--- {key} ---")
    print(value)
    print("\n" + "=" * 50)

## 7. Test Tools Directly

Test the available tools on the dataset without going through the full pipeline.

In [ ]:
# Test tools directly on the dataset
from src.tools import pandas_tools

print("Testing Tools on Dataset:")
print("=" * 50)

# Get row count
print("\n1. Row Count:")
row_count = pandas_tools.get_row_count.invoke({"file_path": DATASET_PATH})
print(f"   {row_count} rows")

# Get column names
print("\n2. Column Names:")
columns = pandas_tools.get_column_names.invoke({"file_path": DATASET_PATH})
print(f"   {columns}")

# Get data types
print("\n3. Data Types:")
dtypes = pandas_tools.get_data_types.invoke({"file_path": DATASET_PATH})
for col, dtype in dtypes.items():
    print(f"   {col}: {dtype}")

# Get missing values
print("\n4. Missing Values:")
missing = pandas_tools.get_missing_values.invoke({"file_path": DATASET_PATH})
for col, count in missing.items():
    print(f"   {col}: {count}")

# Get sample rows
print("\n5. Sample Rows:")
sample = pandas_tools.get_sample_rows.invoke({"file_path": DATASET_PATH})
print(sample)

## Summary

This notebook demonstrated the full metadata extraction pipeline:

1. **Configuration** - Set file type, metadata standard, and execution topology
2. **Exploration** - Viewed available topologies and player roles
3. **Plan Generation** - LLM generated a step-by-step extraction plan
4. **Validation** - Verified plan dataflow dependencies
5. **Dataset Preparation** - Ensured test dataset exists
6. **Full Execution** - Ran parallel players with debate on each step
7. **Tools Testing** - Tested individual tools on the dataset

### Key Components:
- **Orchestrator**: Coordinates planning and execution
- **Players**: Execute tasks and participate in debates  
- **Tools**: Extract actual data from datasets
- **Topology**: Configures parallelism and debate rounds

### Configuration Options:
- Edit `src/config.py` to change the LLM model
- Modify `TOPOLOGY_NAME` to change execution strategy
- Add new player roles in `src/players/configs.py`
- Add new tools in `src/tools/pandas_tools.py`